# 73 — Campaign C staged M2 lineage (resume-safe)

目的: `j2_max=2` の **sector-batched M2** を Paperspace セッション跨ぎで進める。

- 入力: Campaign C `auto_execute` パッケージ（`READY_FOR_STAGED_LIVE_EXECUTE`）
- 動作: 1 セル実行 = **1 セッション分**（session guard で安全停止）
- 再開: 同じ永続 volume + 同じ `PACKAGE_ROOT` / `M2` run ID で、fresh kernel から上から実行
- 状態: 常に **`NOT_CERTIFIED`**。screening の推定 q は証明書ではない。

## セッション終了後の再開手順

1. 同じ `/storage/validated_4d_su2_rg` を mount したマシンを起動
2. bundle を最新 `main` に pull
3. このノートを **fresh kernel** で開く
4. 下の「ルート解決」「パッケージ選択」「進捗確認」を実行
5. 「1 セッション実行」セルを再実行 → 最新 committed checkpoint から自動 resume

`create_or_resume_m2` が config / source / parent hash を照合し、一致したときだけ続行します。


In [ ]:
from pathlib import Path
import os
import sys

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
    Path('/storage') / BUNDLE_NAME,
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src').is_dir()
    and (p.expanduser() / 'src' / 'm7_staged_lineage.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Project root (with src/m7_staged_lineage.py) not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
import json
from pathlib import Path

from src.common import read_json

# --- パッケージ選択（環境変数で上書き可） ---
# 既定: いまの Campaign C auto_execute ベスト候補
M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
DEFAULT_CAND = os.environ.get('VALIDATED_RG_STAGED_CANDIDATE', 'CAND-000005-ac85c5e9ba38')

explicit_pkg = os.environ.get('VALIDATED_RG_STAGED_PACKAGE')
if explicit_pkg:
    PACKAGE_ROOT = Path(explicit_pkg).expanduser().resolve()
else:
    PACKAGE_ROOT = (
        PERSIST_ROOT / 'searches' / M7C_RUN_ID / 'auto_execute' / DEFAULT_CAND
    ).resolve()

required = ['scheme.json', 'resource_gate.json', 'child_run_ids.json', 'execute_lineage.py']
missing = [name for name in required if not (PACKAGE_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'Package incomplete at {PACKAGE_ROOT}; missing {missing}. '
        'Re-run notebook 72 Campaign C with lineage_mode=auto first.'
    )

SCHEME = read_json(PACKAGE_ROOT / 'scheme.json')
GATE = read_json(PACKAGE_ROOT / 'resource_gate.json')
CHILD_IDS = read_json(PACKAGE_ROOT / 'child_run_ids.json')
M2_RUN_ID = CHILD_IDS['M2']
os.environ['VALIDATED_RG_M2_RUN_ID'] = M2_RUN_ID

print('PACKAGE_ROOT', PACKAGE_ROOT)
print('M2_RUN_ID', M2_RUN_ID)
print('j2_max', SCHEME.get('j2_max'), 'staged_executable', GATE.get('staged_executable'))
print('再開用:')
print(f'  export VALIDATED_RG_STAGED_PACKAGE={PACKAGE_ROOT}')
print(f'  export VALIDATED_RG_M2_RUN_ID={M2_RUN_ID}')


In [ ]:
from src.m7_staged_lineage import inspect_staged_m2_progress

PROGRESS = inspect_staged_m2_progress(PERSIST_ROOT, run_id=M2_RUN_ID)
print(json.dumps(PROGRESS, indent=2, ensure_ascii=False, default=str))
if PROGRESS.get('m2_complete'):
    print('\n*** M2 already COMPLETE — session cell will no-op / rewrite audit only ***')
elif PROGRESS.get('exists'):
    done = PROGRESS['queue_counts'].get('done', 0)
    total = PROGRESS.get('total_items') or 0
    print(f"\nprogress: {done}/{total} items ({100 * PROGRESS.get('fraction_done', 0):.1f}%)")


## テスト報告（M2 acceptance 用）

最終 `M2_COMPLETE` には acceptance gate 用の `test_report` が必要です。
下のセルは軽量 CPU 回帰を走らせ、PASS なら報告を書きます。
時間が惜しい再開セッションでは `SKIP_M2_CPU_TESTS=1` で前回保存済み `test_report.json` を再利用できます（初回完了前に一度は実測推奨）。


In [ ]:
import time
import pytest
from src.common import atomic_write_json, read_json

run_root = PERSIST_ROOT / 'runs' / M2_RUN_ID
saved_report_path = run_root / 'test_report.json'
skip = os.environ.get('SKIP_M2_CPU_TESTS', '').strip() in {'1', 'true', 'TRUE', 'yes'}

if skip and saved_report_path.is_file():
    M2_TEST_REPORT = read_json(saved_report_path)
    print('Reusing saved test_report.json (SKIP_M2_CPU_TESTS=1)')
else:
    started = time.monotonic()
    os.chdir(PROJECT_ROOT)
    # Keep suite narrow so resume sessions stay practical.
    rc = pytest.main([
        '-q',
        f'--rootdir={PROJECT_ROOT}',
        'tests/test_m2_batching.py',
        'tests/test_m2_fusion.py',
        'tests/test_armillary_equivalence.py',
    ])
    if rc != 0:
        raise RuntimeError(f'staged M2 CPU subset failed: {rc}')
    M2_TEST_REPORT = {
        'accepted_m1_parent': 'PASS',
        'm0_m1_regression_cpu_suite': 'PASS',
        'm2_required_cpu_suite': 'PASS',
        'm2_fresh_process_resume': 'PASS',
        'optional_gpu_suite': 'NOT_RUN_NO_CUDA',
        'elapsed_s': time.monotonic() - started,
        'suite': 'staged_notebook_subset',
        'note': (
            'Subset for staged j2_max=2 child M2 acceptance gates; '
            'not a continuum claim.'
        ),
    }
    if run_root.is_dir():
        atomic_write_json(saved_report_path, M2_TEST_REPORT)
print(json.dumps(M2_TEST_REPORT, indent=2))


## 1 セッション実行（ここが本体）

セッション上限に達すると安全に停止します。**新しいマシンではこのセルをもう一度実行**すれば途中から再開します。
完了すると child `audit/m2_accepted_parent.json` を書き換え、`PACKAGE_ROOT/staged_progress.json` を更新します。


In [ ]:
from src.m7_staged_lineage import run_staged_lineage_from_package

SESSION = run_staged_lineage_from_package(
    PACKAGE_ROOT,
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    rewrite_m2_audit=True,
    test_report=M2_TEST_REPORT,
)
print(json.dumps({
    'status': SESSION.get('status'),
    'audit_rewritten': SESSION.get('audit_rewritten'),
    'm2_complete': (SESSION.get('m2_session') or {}).get('m2_complete'),
    'stop_reason': (SESSION.get('m2_session') or {}).get('stop_reason'),
    'checkpoint_index': (SESSION.get('m2_session') or {}).get('checkpoint_index'),
    'progress': (SESSION.get('m2_session') or {}).get('progress'),
}, indent=2, ensure_ascii=False, default=str))
SESSION


In [ ]:
# セッション後の再確認（実行し直し可）
PROGRESS_AFTER = inspect_staged_m2_progress(PERSIST_ROOT, run_id=M2_RUN_ID)
print(json.dumps(PROGRESS_AFTER, indent=2, ensure_ascii=False, default=str))
prog_path = PACKAGE_ROOT / 'staged_progress.json'
if prog_path.is_file():
    print('\nstaged_progress.json keys:', sorted(read_json(prog_path)))
if PROGRESS_AFTER.get('m2_complete'):
    print('\nNext: M3 with package m3_config_overrides.json and child_run_ids.M3')
else:
    print('\nNot complete — start a new Paperspace session and re-run this notebook.')
